# Análise de Explicabilidade (XAI) e Discussão dos Resultados

Este notebook consolida a etapa de análise interpretativa do modelo CNN-LSTM aplicado à manutenção prescritiva no contexto da Indústria 4.0. 

As perguntas à serem respondidas nesse notabook são:

1. **Quais sensores dominam a previsão?**

2. **Como essa importância muda ao longo do tempo?**

3. **Existe mudança de importância conforme a complexidade do dataset?**

4. **Existe consenso entre diferentes modelos?**

5. **Avaliando a estabilidade: Treinando o modelo 30 vezes, as explicações permanecem iguais?**

6. **Quando o modelo erra muito, qual sensor causou isso?**


Para responder a essas questões, utilizaremos o pipeline modular construído no diretório `src/xai/`, que integra métodos de atribuição de importância (SHAP, TimeSHAP, WindowSHAP) e métodos baseados em gradientes (Grad-CAM e Saliency Maps).

In [17]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np


sys.path.append(os.path.abspath('../..')) 

from src.xai.xai_baseExplainer import BaseExplainer
from src.xai.xai_shapExplainer import ShapExplainer
from src.xai.xai_limeExplainer import LimeExplainer
from src.xai.xai_igExplainer import IntegratedGradientsExplainer
from src.xai.xai_timeshapExplainer import TimeShapExplainer
from src.xai.xai_windowshapExplainer import WindowShapExplainer
from src.xai.xaI_gradcamExplainer import GradCamExplainer
from src.xai.xai_saliencymapsExplainer import SaliencyMapsExplainer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
DATASET_01 = "FD001"
model_path_001 = f"../../models/best_models_cnn_lstm/best_cnn_lstm_{DATASET_01}.keras"
x_path_001 = f"../../data/processed_data/X_test_{DATASET_01}.npz"
y_path_001 = f"../../data/processed_data/y_test_{DATASET_01}.npz"

DATASET_02 = "FD002"
model_path_002 = f"../../models/best_models_cnn_lstm/best_cnn_lstm_{DATASET_02}.keras"
x_path_002 = f"../../data/processed_data/X_test_{DATASET_02}.npz"
y_path_002 = f"../../data/processed_data/y_test_{DATASET_02}.npz"

DATASET_03 = "FD003"
model_path_003 = f"../../models/best_models_cnn_lstm/best_cnn_lstm_{DATASET_03}.keras"
x_path_003 = f"../../data/processed_data/X_test_{DATASET_03}.npz"
y_path_003 = f"../../data/processed_data/y_test_{DATASET_03}.npz"

DATASET_04 = "FD004"
model_path_004 = f"../../models/best_models_cnn_lstm/best_cnn_lstm_{DATASET_04}.keras"
x_path_004 = f"../../data/processed_data/X_test_{DATASET_04}.npz"
y_path_004 = f"../../data/processed_data/y_test_{DATASET_04}.npz"

## 1. Quais sensores dominam a previsão?

Para responder a esta pergunta, aplicaremos os métodos SHAP, Saliency Maps e Integrated Gradients. 

O objetivo é cruzar os resultados abordagens independentes de XAI, se os métodos convergirem para o mesmo grupo de sensores, teremos uma forte evidência de que essas variáveis são, de fato, os principais indicadores físicos da degradação do motor aprendidos pela CNN-LSTM. Utilizando três métodos facilita caso aconteça um empate no sensor vencedor.

A ideia foi criar uma tabela, com um ranking dos sensores que mais impactam em cada método de XAI, de forma que no final é gerado um consenso com os sensores mais impactantes considerando todos métodos de XAI

In [19]:
shap_explainer_001 = ShapExplainer(model_path_001, x_path_001, y_path_001, background_size=100)
saliency_explainer_001 = SaliencyMapsExplainer(model_path_001, x_path_001, y_path_001)
ig_explainer_001 = IntegratedGradientsExplainer(model_path_001, x_path_001, y_path_001)

#shap_explainer.plot_global_summary(num_samples=100)

#saliency_explainer.plot_global_summary(num_samples=100)

#ig_explainer.plot_global_summary(num_samples=100)

In [20]:
def gerar_tabela_consenso(shap_explainer, saliency_explainer, ig_explainer, feature_names, num_samples=100):
    print(f"Extraindo dados de {num_samples} amostras para a Tabela de Consenso (aguarde)...")
    
    n_sensores = shap_explainer.X_test.shape[2]
    
    shap_importances = np.zeros(n_sensores)
    saliency_importances = np.zeros(n_sensores)
    ig_importances = np.zeros(n_sensores)
           
    for i in range(num_samples):

        sv = shap_explainer.explain_instance(i)
        if sv.ndim == 4:
            sv = sv[0, :, :, 0]
        shap_importances += np.sum(np.abs(sv), axis=0)

        sal = saliency_explainer.explain_instance(i)
        saliency_importances += np.sum(sal, axis=0) 

        ig = ig_explainer.explain_instance(i)
        ig_importances += np.sum(np.abs(ig), axis=0)

    def get_top10_scores(importances):
        top_indices = np.argsort(importances)[-10:][::-1]
        scores = {}
        ranking = {}
        for rank, sensor_idx in enumerate(top_indices):
            nome_sensor = feature_names[sensor_idx] 
            scores[nome_sensor] = 10 - rank
            ranking[nome_sensor] = rank + 1
        return scores, ranking

    shap_scores, shap_rank = get_top10_scores(shap_importances)
    sal_scores, sal_rank = get_top10_scores(saliency_importances)
    ig_scores, ig_rank = get_top10_scores(ig_importances)

    consenso = {}

    for sensor in feature_names:
        score_total = shap_scores.get(sensor, 0) + sal_scores.get(sensor, 0) + ig_scores.get(sensor, 0)
        
        if score_total > 0:
            consenso[sensor] = {
                "Pontuação Final": score_total,
                "Ranking (SHAP)": shap_rank.get(sensor, "-"),
                "Ranking (Saliency)": sal_rank.get(sensor, "-"),
                "Ranking (IG)": ig_rank.get(sensor, "-")
            }

    df_consenso = pd.DataFrame.from_dict(consenso, orient='index')
    if not df_consenso.empty:
        df_consenso = df_consenso.sort_values(by="Pontuação Final", ascending=False).head(5)
        
    return df_consenso

In [21]:
nomes_reais_fd001 = np.load(x_path_001)['features']

gerar_tabela_consenso(shap_explainer_001, saliency_explainer_001, ig_explainer_001, feature_names=nomes_reais_fd001, num_samples=200)

Extraindo dados de 200 amostras para a Tabela de Consenso (aguarde)...


C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_30
Received: inputs=['Tensor(shape=(1, 30, 19))']
  warnings.warn(msg)
C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_30
Received: inputs=['Tensor(shape=(50, 30, 19))']
  warnings.warn(msg)


,Pontuação Final,Ranking (SHAP),Ranking (Saliency),Ranking (IG)
s_11,29,1,1,2
s_12,27,2,3,1
s_15,22,3,4,4
s_7,21,4,5,3
s_4,16,5,6,6


In [22]:
shap_explainer_002 = ShapExplainer(model_path_002, x_path_002, y_path_002, background_size=100)
saliency_explainer_002 = SaliencyMapsExplainer(model_path_002, x_path_002, y_path_002)
ig_explainer_002 = IntegratedGradientsExplainer(model_path_002, x_path_002, y_path_002)

nomes_reais_fd002 = np.load(x_path_002)['features']

gerar_tabela_consenso(shap_explainer_002, saliency_explainer_002, ig_explainer_002, feature_names=nomes_reais_fd002, num_samples=200)

Extraindo dados de 200 amostras para a Tabela de Consenso (aguarde)...


C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_61
Received: inputs=['Tensor(shape=(1, 30, 24))']
  warnings.warn(msg)
C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_61
Received: inputs=['Tensor(shape=(50, 30, 24))']
  warnings.warn(msg)


,Pontuação Final,Ranking (SHAP),Ranking (Saliency),Ranking (IG)
s_11,30,1,1,1
s_15,27,2,2,2
s_4,24,3,3,3
s_2,19,4,5,5
s_7,18,7,4,4


In [23]:
shap_explainer_003 = ShapExplainer(model_path_003, x_path_003, y_path_003, background_size=100)
saliency_explainer_003 = SaliencyMapsExplainer(model_path_003, x_path_003, y_path_003)
ig_explainer_003 = IntegratedGradientsExplainer(model_path_003, x_path_003, y_path_003)

nomes_reais_fd003 = np.load(x_path_003)['features']

gerar_tabela_consenso(shap_explainer_003, saliency_explainer_003, ig_explainer_003, feature_names=nomes_reais_fd003, num_samples=200)

Extraindo dados de 200 amostras para a Tabela de Consenso (aguarde)...


C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_92
Received: inputs=['Tensor(shape=(1, 30, 20))']
  warnings.warn(msg)
C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_92
Received: inputs=['Tensor(shape=(50, 30, 20))']
  warnings.warn(msg)


,Pontuação Final,Ranking (SHAP),Ranking (Saliency),Ranking (IG)
s_12,29,1,1,2
s_7,26,2,2,3
s_6,25,4,3,1
s_13,22,3,4,4
s_8,18,5,5,5


In [24]:
shap_explainer_004 = ShapExplainer(model_path_004, x_path_004, y_path_004, background_size=100)
saliency_explainer_004 = SaliencyMapsExplainer(model_path_004, x_path_004, y_path_004)
ig_explainer_004 = IntegratedGradientsExplainer(model_path_004, x_path_004, y_path_004)

nomes_reais_fd004 = np.load(x_path_004)['features']

gerar_tabela_consenso(shap_explainer_004, saliency_explainer_004, ig_explainer_004, feature_names=nomes_reais_fd004, num_samples=200)

Extraindo dados de 200 amostras para a Tabela de Consenso (aguarde)...


C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_123
Received: inputs=['Tensor(shape=(1, 30, 24))']
  warnings.warn(msg)
C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_123
Received: inputs=['Tensor(shape=(50, 30, 24))']
  warnings.warn(msg)


,Pontuação Final,Ranking (SHAP),Ranking (Saliency),Ranking (IG)
s_9,30,1,1,1
s_14,26,2,3,2
s_12,23,3,4,3
s_11,22,5,2,4
s_7,19,4,5,5
